In [1]:
import os

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore

C:\Users\bhawa\AppData\Local\Temp\ipykernel_24516\2619958126.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


In [2]:
loader = PyMuPDFLoader("data/QUANTUM MECHANICS.pdf")

documents = loader.load()

print("Number of pages:", len(documents))

Number of pages: 8


In [3]:
print(documents[0].page_content[:200])

QUANTUM MECHANICS 
1. Introduction 
Quantum Mechanics is a branch of physics that deals with the behavior of matter and 
energy at very small scales, such as atoms, electrons, photons, and other micro


In [4]:
print(documents[0].metadata)

{'producer': 'Microsoft® Word 2024', 'creator': 'Microsoft® Word 2024', 'creationdate': '2026-09-30T09:31:46+05:30', 'source': 'data/QUANTUM MECHANICS.pdf', 'file_path': 'data/QUANTUM MECHANICS.pdf', 'total_pages': 8, 'format': 'PDF 1.7', 'title': '', 'author': 'Bhawani Kumari', 'subject': '', 'keywords': '', 'moddate': '2026-09-30T09:31:46+05:30', 'trapped': '', 'modDate': "D:20260930093146+05'30'", 'creationDate': "D:20260930093146+05'30'", 'page': 0}


In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print("Number of chunks:", len(chunks))

Number of chunks: 21


In [6]:
print(chunks[0].page_content)

QUANTUM MECHANICS 
1. Introduction 
Quantum Mechanics is a branch of physics that deals with the behavior of matter and 
energy at very small scales, such as atoms, electrons, photons, and other microscopic 
particles. Classical mechanics, developed mainly by Newton, successfully explains the 
motion of everyday objects, but it cannot properly explain many phenomena occurring 
at the atomic and subatomic levels. 
Quantum mechanics introduced a new way of understanding nature in which physical


In [7]:
print(chunks[0].metadata)

{'producer': 'Microsoft® Word 2024', 'creator': 'Microsoft® Word 2024', 'creationdate': '2026-09-30T09:31:46+05:30', 'source': 'data/QUANTUM MECHANICS.pdf', 'file_path': 'data/QUANTUM MECHANICS.pdf', 'total_pages': 8, 'format': 'PDF 1.7', 'title': '', 'author': 'Bhawani Kumari', 'subject': '', 'keywords': '', 'moddate': '2026-09-30T09:31:46+05:30', 'trapped': '', 'modDate': "D:20260930093146+05'30'", 'creationDate': "D:20260930093146+05'30'", 'page': 0}


In [8]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [9]:
vector = embeddings.embed_query("What is Heisenberg Uncertainty Principle?")

print(len(vector))

384


In [10]:
import pinecone
print(pinecone.__version__)

6.0.2


In [11]:
import os
from dotenv import load_dotenv
from pinecone import Pinecone

load_dotenv()

pc = Pinecone(
    api_key=os.getenv("PINECONE_API_KEY")
)

print(pc.list_indexes())

[{
    "name": "academic-rag",
    "metric": "cosine",
    "host": "academic-rag-hz1j223.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null
}, {
    "name": "ragproject",
    "metric": "cosine",
    "host": "ragproject-hz1j223.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 1024,
    "deletion_protection": "disabled",
    "tags": null,
    "embed": {
        "model": "llama-text-embed-v2",
        "field_map": {
            "text": "text"
        },
        "dimension": 1024,
        "metric": "cosine

In [12]:
vector_store = PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    index_name="academic-rag"
)

print("Uploaded successfully!")

Uploaded successfully!


In [13]:
question = "What is quantum mechanics?"

results = vector_store.similarity_search(
    question,
    k=3
)

print("Number of results:", len(results))

Number of results: 3


In [14]:
for i, doc in enumerate(results):
    print(f"\n{'='*60}")
    print(f"RESULT {i+1}")
    print(f"{'='*60}")
    
    print(doc.page_content)
    print("\nMetadata:")
    print(doc.metadata)


RESULT 1
QUANTUM MECHANICS 
1. Introduction 
Quantum Mechanics is a branch of physics that deals with the behavior of matter and 
energy at very small scales, such as atoms, electrons, photons, and other microscopic 
particles. Classical mechanics, developed mainly by Newton, successfully explains the 
motion of everyday objects, but it cannot properly explain many phenomena occurring 
at the atomic and subatomic levels. 
Quantum mechanics introduced a new way of understanding nature in which physical

Metadata:
{'author': 'Bhawani Kumari', 'creationDate': "D:20260930093146+05'30'", 'creationdate': '2026-09-30T09:31:46+05:30', 'creator': 'Microsoft® Word 2024', 'file_path': 'data/QUANTUM MECHANICS.pdf', 'format': 'PDF 1.7', 'keywords': '', 'modDate': "D:20260930093146+05'30'", 'moddate': '2026-09-30T09:31:46+05:30', 'page': 0.0, 'producer': 'Microsoft® Word 2024', 'source': 'data/QUANTUM MECHANICS.pdf', 'subject': '', 'title': '', 'total_pages': 8.0, 'trapped': ''}

RESULT 2
10. Concl

In [15]:
questions = [
    "What is quantum mechanics?",
    "What is the uncertainty principle?",
    "Explain wave-particle duality.",
    "What is the Schrödinger equation?",
    "What are quantum numbers?"
]

In [16]:
for question in questions:
    
    print(f"\n\nQUESTION: {question}")
    
    results = vector_store.similarity_search(
        question,
        k=3
    )
    
    for i, doc in enumerate(results):
        print(f"\n--- Result {i+1} ---")
        print(doc.page_content[:500])
        print("Source:", doc.metadata)



QUESTION: What is quantum mechanics?

--- Result 1 ---
QUANTUM MECHANICS 
1. Introduction 
Quantum Mechanics is a branch of physics that deals with the behavior of matter and 
energy at very small scales, such as atoms, electrons, photons, and other microscopic 
particles. Classical mechanics, developed mainly by Newton, successfully explains the 
motion of everyday objects, but it cannot properly explain many phenomena occurring 
at the atomic and subatomic levels. 
Quantum mechanics introduced a new way of understanding nature in which physical
Source: {'author': 'Bhawani Kumari', 'creationDate': "D:20260930093146+05'30'", 'creationdate': '2026-09-30T09:31:46+05:30', 'creator': 'Microsoft® Word 2024', 'file_path': 'data/QUANTUM MECHANICS.pdf', 'format': 'PDF 1.7', 'keywords': '', 'modDate': "D:20260930093146+05'30'", 'moddate': '2026-09-30T09:31:46+05:30', 'page': 0.0, 'producer': 'Microsoft® Word 2024', 'source': 'data/QUANTUM MECHANICS.pdf', 'subject': '', 'title': '', 'total_pag

In [18]:
from dotenv import load_dotenv
import os

load_dotenv()

print(os.getenv("GROQ_API_KEY") is not None)

True


In [32]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

In [33]:
from langchain_core.prompts import ChatPromptTemplate

In [38]:
prompt = ChatPromptTemplate.from_template("""
You are an academic assistant.

Answer the question using ONLY the provided context.

Instructions:
- Give a clear and concise academic answer.
- Explain the concept in simple language.
- Use bullet points when appropriate.
- Use normal text for mathematical expressions where possible.
- Do not invent information.
- If the answer is not present in the context, say:
  "I couldn't find this information in the provided study material."

Context:
{context}

Question:
{question}

Answer:
""")

In [39]:
def ask_question(question):
    
    results = vector_store.similarity_search(
        question,
        k=3
    )
    
    context = "\n\n".join(
        doc.page_content for doc in results
    )
    
    messages = prompt.invoke({
        "context": context,
        "question": question
    })
    
    response = llm.invoke(messages)
    
    return response.content, results

In [40]:
answer, sources = ask_question(
    "What is the uncertainty principle?"
)

print("ANSWER:")
print(answer)

print("\n\nSOURCES:")

for i, doc in enumerate(sources):
    print(f"\nSource {i+1}")
    print("File:", doc.metadata.get("source"))
    print("Page:", doc.metadata.get("page"))

ANSWER:
**Answer**

The Heisenberg Uncertainty Principle is a fundamental rule of quantum mechanics that says you cannot know a microscopic particle’s position and momentum at the same time with arbitrary precision.  

- **Mathematical form**  
  \[
  \Delta x \, \Delta p \;\ge\; \frac{\hbar}{2}
  \]
  where  
  * Δx = uncertainty in position  
  * Δp = uncertainty in momentum  
  * ℏ = h/2π (reduced Planck constant)

- **Key points**  
  * It is not a limitation of measurement tools; it reflects an intrinsic property of particles.  
  * If you measure an electron’s position very accurately (small Δx), its momentum becomes highly uncertain (large Δp).  
  * Conversely, measuring momentum accurately makes the position uncertain.  
  * This principle distinguishes quantum mechanics from classical physics.


SOURCES:

Source 1
File: data/QUANTUM MECHANICS.pdf
Page: 1.0

Source 2
File: data/QUANTUM MECHANICS.pdf
Page: 2.0

Source 3
File: data/QUANTUM MECHANICS.pdf
Page: 1.0


In [41]:
answer, sources = ask_question(
    "What is the capital of France?"
)

print(answer)

I couldn't find this information in the provided study material.


In [42]:
questions = [
    "What is quantum mechanics?",
    "What is the uncertainty principle?",
    "Explain wave-particle duality.",
    "What is the Schrödinger equation?",
    "What are quantum numbers?",
    "What is the capital of France?"
]

In [43]:
for question in questions:
    answer, sources = ask_question(question)
    
    print(f"\n\nQUESTION: {question}")
    print("ANSWER:")
    print(answer)
    
    print("\nSOURCES:")
    for i, doc in enumerate(sources):
        print(f"\nSource {i+1}")
        print("File:", doc.metadata.get("source"))
        print("Page:", doc.metadata.get("page"))



QUESTION: What is quantum mechanics?
ANSWER:
**Answer**

Quantum mechanics is a branch of physics that describes the behavior of matter and energy at microscopic scales (atoms, electrons, photons, etc.). It replaces classical mechanics for systems where classical laws fail and introduces several foundational concepts:

- **Wave‑particle duality** – particles exhibit both wave‑like and particle‑like properties.
- **Uncertainty principle** – limits the precision with which certain pairs of physical properties (e.g., position and momentum) can be known simultaneously.
- **Wave function** – a mathematical function that encodes the probabilities of a system’s measurable properties.
- **Schrödinger equation** – the fundamental equation governing the time evolution of the wave function.
- **Quantization** – energy and other physical quantities can take only discrete values.

These ideas form the mathematical framework that explains experimental observations and predicts the behavior of micr